<a href="https://colab.research.google.com/github/suyogy1-bot/Machine_Learning/blob/main/Decision_Tree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Student Placement Prediction using a Decision Tree

In this notebook we build a complete machine learning project step by step.
We use simple code and explain every step in simple English.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

---
## Step 1: Problem

**Real-life story:** A college placement cell wants to know which students are likely to get a job placement.
If they know this early, they can give extra help (mock interviews, practice tests, mentoring) to the students who need it.

**What we are predicting:** `placement_status` (Placed or Not Placed)

**Type of problem:** Classification, because the answer is one of two classes.

**What we use to predict (features):**
- Student details: gender, branch, extracurricular activities
- Habits: study hours, sleep hours, internet usage
- Performance: attendance, assignments completed, previous score, exam score

**Two types of mistakes the model can make**
- It says "Placed" but the student is not placed: the student may miss the extra help.
- It says "Not Placed" but the student is placed: the college wastes some effort.

---
## Step 2: Data + EDA (Find patterns)

EDA means **Exploratory Data Analysis**.
It is like meeting a new student for the first time: first you look, ask questions and understand, then you decide what to do.

### 2.1 Load the data

In [ ]:
df = pd.read_csv("/content/student_performance_dataset.csv")

In [ ]:
df.head()

,student_id,gender,branch,study_hours,attendance,sleep_hours,internet_usage,assignments_completed,previous_score,extracurricular,exam_score,placement_status
0,STU01978,Female,Mechanical,1.5,68.2,6.7,2.6,15.0,63.6,Yes,60.8,Placed
1,STU03881,Male,Mechanical,5.0,80.9,5.5,6.2,12.0,55.4,No,58.8,Not Placed
2,STU00053,Female,CSE,0.8,69.4,7.4,3.6,NaN,43.2,No,44.0,Not Placed
3,STU02552,Female,Commerce,1.3,77.1,7.4,3.6,NaN,76.9,Yes,55.5,Placed
4,STU02247,Male,IT,0.0,46.8,7.4,5.6,4.0,49.7,Yes,41.6,Not Placed


### 2.2 Understand the columns

| Column | Meaning |
|---|---|
| `student_id` | Unique ID of the student (only a label, not useful for prediction) |
| `gender` | Male or Female |
| `branch` | CSE, IT, ECE, Mechanical, Civil, Commerce |
| `study_hours` | Study hours per day |
| `attendance` | Attendance in percent |
| `sleep_hours` | Sleep hours per day |
| `internet_usage` | Internet use in hours per day |
| `assignments_completed` | Number of assignments completed |
| `previous_score` | Marks in the previous exam |
| `extracurricular` | Does the student take part in extra activities? |
| `exam_score` | Latest exam marks |
| `placement_status` | **Target**: Placed or Not Placed |

In [ ]:
# Data types and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   student_id             10000 non-null  object 
 1   gender                 10000 non-null  object 
 2   branch                 9901 non-null   object 
 3   study_hours            9699 non-null   float64
 4   attendance             9600 non-null   float64
 5   sleep_hours            9702 non-null   float64
 6   internet_usage         9696 non-null   float64
 7   assignments_completed  9751 non-null   float64
 8   previous_score         9652 non-null   float64
 9   extracurricular        10000 non-null  object 
 10  exam_score             10000 non-null  float64
 11  placement_status       10000 non-null  object 
dtypes: float64(7), object(5)
memory usage: 937.6+ KB


In [ ]:
# Quick summary of the number columns
df.describe()

,study_hours,attendance,sleep_hours,internet_usage,assignments_completed,previous_score,exam_score
count,9699.000000,9600.000000,9702.000000,9696.000000,9751.000000,9652.000000,10000.000000
mean,3.487277,75.787323,6.804556,4.495153,12.835812,61.782553,60.555980
std,1.630787,10.962674,1.114314,1.687334,4.129272,13.031526,14.234672
min,0.000000,33.900000,1.000000,0.200000,0.000000,20.000000,0.000000
25%,2.400000,68.400000,6.100000,3.400000,10.000000,52.800000,50.700000
50%,3.400000,75.700000,6.800000,4.500000,13.000000,61.900000,60.500000
75%,4.500000,83.400000,7.600000,5.600000,16.000000,70.700000,70.100000
max,21.300000,100.000000,14.000000,21.500000,20.000000,100.000000,100.000000


In [ ]:
# How many missing values are in each column?
df.isnull().sum()

,0
student_id,0
gender,0
branch,99
study_hours,301
attendance,400
sleep_hours,298
internet_usage,304
assignments_completed,249
previous_score,348
extracurricular,0


In [ ]:
# Are there duplicate rows?
df.duplicated().sum()

np.int64(50)

In [ ]:
# check Columns purity
cat_col = df.select_dtypes(include='object').columns
cat_col


Index(['student_id', 'gender', 'branch', 'extracurricular',
       'placement_status'],
      dtype='object')

In [ ]:
for col in cat_col:
  print(col)
  print(df[col].value_counts())
  print()

student_id
student_id
STU07967    2
STU05972    2
STU01267    2
STU00660    2
STU02228    2
           ..
STU04456    1
STU08594    1
STU01060    1
STU03607    1
STU03304    1
Name: count, Length: 9950, dtype: int64

gender
gender
Male      5069
Female    4779
M           42
female      40
male        36
F           34
Name: count, dtype: int64

branch
branch
CSE             2671
IT              1723
ECE             1561
Mechanical      1337
Commerce        1316
Civil           1060
 CSE              29
cse               29
it                25
commerce          18
 Commerce         18
MECHANICAL        15
 IT               15
mechanical        14
 Civil            14
 ECE              12
COMMERCE          11
 Mechanical       11
CIVIL              8
ece                7
civil              7
Name: count, dtype: int64

extracurricular
extracurricular
No     6046
Yes    3954
Name: count, dtype: int64

placement_status
placement_status
Not Placed    5116
Placed        4884
Name: count, dt

### What is wrong with this data?

Real-world data is never perfect. Here we can see:

1. **Missing values** in many columns (study hours, attendance, sleep hours, and more).
2. **Duplicate rows**, the same student appears twice.
3. **Gender is written in many ways**: `Male`, `male`, `M`, `Female`, `female`, `F`.
4. **Branch has spelling problems**: `CSE`, `cse`, ` CSE ` (with extra spaces), `MECHANICAL`, `mechanical`.
5. **Impossible values (outliers)**: check `study_hours`, `sleep_hours` and `internet_usage` in the table above.
   Nobody can study 21 hours every day!

We will fix these in Step 3 and Step 4.

### 2.3 Is our target balanced?

### 2.4 Do placed students look different?
Let us compare the average of every number column for the two groups.

In [ ]:
# Group by num_col to placement_status



In [ ]:
# Box plots: compare Placed and Not Placed for each number column


In [ ]:
# Correlation: which number columns move together?


---
## Step 3: Clean data

Rule: **never change the original data**. We work on a copy, so we can always go back.

We will do 5 cleaning jobs:
1. Remove duplicate rows
2. Drop `student_id` (it is only a label)
3. Fix the `gender` column
4. Fix the `branch` column
5. Fix impossible values (outliers)



In [ ]:
# Make a copy so the original df stays safe


# Job 1: remove duplicate rows

# Job 1.0: Remove null values

In [ ]:
# Job 2: student_id is only a label. It does not help to predict placement.


In [ ]:
# Job 3: fix gender
# strip() removes extra spaces, lower() makes small letters, map() gives one clean name


In [ ]:
# Job 4: fix branch
# Remove extra spaces and make everything CAPITAL letters, so ' cse ' becomes 'CSE'


In [ ]:
# Job 5: fix impossible values with clip()
# clip(0, 12) means: anything above 12 becomes 12, anything below 0 becomes 0

# df_clean["study_hours"]
# df_clean["sleep_hours"]
# df_clean["internet_usage"]



In [ ]:
# Quick look at the cleaned categories


Cleaning is done. Values are now consistent and there are no impossible numbers.
The missing values are still there. We fix them next.